# Task 2 — Enhanced Strategy


## Overview And Research Purpose
This notebook documents the Task 2 enhanced commodity trend-following strategy. It runs the enhanced pipeline with multi-lookback conviction, optional macro modulation, execution-time short RSI cap, Parabolic SAR exits, conviction sizing, and transaction costs. SAR and cost diagnostics are training-only; final holdout evaluation belongs at the end after parameters are frozen.


## 0. Imports And Data Loading
Set up local imports, load training prices and returns, and load training-period macro inputs used for conviction modulation.


In [ ]:
%load_ext autoreload
%autoreload 2

import sys, os, warnings
from pathlib import Path
warnings.filterwarnings('ignore')

ROOT = Path(os.path.abspath('')).parents[1]
sys.path.insert(0, str(ROOT))                         # tasks.task_1.* / tasks.task_2.* qualified imports
sys.path.insert(0, str(ROOT / 'tasks' / 'task_2'))   # strategy_enhanced, signals, portfolio
sys.path.insert(0, str(ROOT / 'tasks' / 'task_1'))   # indicators.py, strategy (baseline), analytics

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

plt.rcParams['figure.dpi'] = 120
plt.rcParams['axes.grid']  = True
plt.rcParams['grid.alpha'] = 0.25

from strategy_enhanced import build_signals as build_signals_enh, run_strategy as run_strategy_enh
from diagnostics      import transaction_cost_sensitivity, sl_mult_grid_search, sar_grid_search
from strategy          import run_strategy  as run_strategy_base
from analytics         import (performance_stats, sector_contribution,
                                position_summary, plot_signals,
                                rolling_sharpe, plot_rolling_sharpe)


In [ ]:
prices = pd.read_csv(ROOT / 'data/training/close_prices_final.csv',
                     index_col=0, parse_dates=True).sort_index()

returns = prices.pct_change()
assets  = pd.read_csv(ROOT / 'data/raw/assets.csv')
factor  = returns.mean(axis=1)
factor.name = 'EW Factor'

print(f'Period      : {prices.index[0].date()} → {prices.index[-1].date()}')
print(f'Commodities : {prices.shape[1]}')
print(f'Days        : {len(prices):,}')


In [ ]:
# ── Load external macro data (aligned to training period) ──────────────────
ext_dir = ROOT / 'data/external/training'

fx_spot  = pd.read_csv(ext_dir / 'fx_spot_levels.csv',
                       index_col=0, parse_dates=True).sort_index()
equity   = pd.read_csv(ext_dir / 'equity_indices_levels.csv',
                       index_col=0, parse_dates=True).sort_index()
credit   = pd.read_csv(ext_dir / 'credit_spreads_levels.csv',
                       index_col=0, parse_dates=True).sort_index()
fx_vol   = pd.read_csv(ext_dir / 'atm_vols_levels.csv',
                       index_col=0, parse_dates=True).sort_index()

external = {'fx_spot': fx_spot, 'equity': equity, 'credit': credit, 'fx_vol': fx_vol}

print(f'FX spot    : {fx_spot.shape}   cols: {list(fx_spot.columns[:4])}...')
print(f'Equity     : {equity.shape}    cols: {list(equity.columns[:4])}')
print(f'Credit     : {credit.shape}    cols: {list(credit.columns)}')
print(f'FX vol     : {fx_vol.shape}    cols: {list(fx_vol.columns[:4])}...')


### 0.1 Macro Inputs And Signals
Build the dollar, equity, credit, and volatility diagnostics used by the enhanced strategy. These signals modulate conviction only; they do not change trade direction.


In [ ]:
from signals.macro_signal import (dollar_signal, equity_signal, credit_signal,
                                   vol_percentile, macro_regime_score)

d_sig = dollar_signal(fx_spot,  fast=50, slow=200)
e_sig = equity_signal(equity,   fast=50, slow=200)
c_sig = credit_signal(credit,   fast=50, slow=200)
v_pct = vol_percentile(fx_vol)
m_score = macro_regime_score(dollar=d_sig, equity=e_sig, credit=c_sig)

# Align to prices index
d_sig   = d_sig.reindex(prices.index)
e_sig   = e_sig.reindex(prices.index)
c_sig   = c_sig.reindex(prices.index)
v_pct   = v_pct.reindex(prices.index)
m_score = m_score.reindex(prices.index).fillna(0)

fig, axes = plt.subplots(5, 1, figsize=(16, 16), sharex=True)
fig.suptitle('Macro Regime Signals  |  Aligned to Commodity Training Period',
             fontweight='bold', fontsize=12)

panel_data = [
    (d_sig,   'Dollar Weakness (+1 = weak USD = bullish for commodities)',    'steelblue'),
    (e_sig,   'SPX Trend (+1 = risk-on uptrend)',                             'darkgreen'),
    (c_sig,   'Credit Risk (+1 = CDX HY spreads tightening = risk-on)',       'darkorange'),
    (m_score, 'Macro Score (mean of above, continuous [-1, +1])',             'black'),
    (v_pct,   'EUR/USD 3M ATM Vol Percentile  (>0.80 = high-vol regime cap)', 'firebrick'),
]
for ax, (s, title, col) in zip(axes, panel_data):
    ax.plot(s.index, s.values, color=col, lw=1.2)
    if s.abs().max() <= 1.05:   # signal series
        ax.axhline(0,  color='grey', lw=0.5)
        ax.axhline( 0.80, color='grey', lw=0.4, linestyle=':')
        ax.axhline(-0.80, color='grey', lw=0.4, linestyle=':')
    ax.set_title(title, fontsize=9, loc='left')
    ax.spines[['top', 'right']].set_visible(False)

fig.tight_layout()
plt.show()


### 0.2 Baseline Comparison Setup
Run the Task 1 baseline once so the enhanced charts and tables can compare against the original strategy.


In [ ]:
# Run baseline strategy (Task 1) — needed for comparison charts
from strategy import run_strategy as _run_base

positions, weights, port_r = _run_base(
    prices,
    returns        = returns,
    filter_fast    = 50,
    filter_slow    = 200,
    slope_lookback = 100,
    ema_window     = 14,
    rsi_window     = 14,
    rsi_long_level = 50.0,
    rsi_level      = 60.0,
    rsi_flag_window= 5,
    bb_window      = 100,
    bb_num_std     = 2.0,
    atr_window     = 50,
    sl_mult        = 3.0,
    risk_per_trade = 0.01,
    max_weight     = 0.20,
    leverage_cap   = 2.50,
)


## 1. Control Panel
Central parameter block for the enhanced strategy. Keep the RSI short cap, SAR settings, macro sizing, and cost assumptions visible here before running the pipeline.


In [ ]:
# ── Layer 1 — Regime filter (Enhanced) ────────────────────────────────────────
ENH_FILTER_FAST = 50
ENH_FILTER_SLOW = 200
# Lookbacks default: [20, 30, 40, 50, 60, 70, 80, 90, 100, 110]

# ── Layer 2 — Entry timing (Enhanced) ─────────────────────────────────────────
ENH_EMA_WINDOW      = 14
ENH_RSI_WINDOW      = 14
ENH_RSI_LONG_LEVEL  = 50.0
ENH_RSI_LEVEL       = 60.0
ENH_RSI_FLAG_WINDOW = 5
ENH_RSI_SHORT_CAP   = 50.0   # short blocked if RSI < this at execution bar

# ── Slot manager — Parabolic SAR ───────────────────────────────────────────────
ENH_BB_WINDOW        = 100
ENH_BB_NUM_STD       = 2.0
ENH_ATR_WINDOW       = 50
ENH_SAR_INITIAL_MULT = 2.5
ENH_SAR_AF_START     = 0.01
ENH_SAR_AF_STEP      = 0.03
ENH_SAR_AF_MAX       = 0.20
ENH_SAR_GRACE_PERIOD = 5

# ── Portfolio — conviction-weighted sizing ─────────────────────────────────────
ENH_SL_MULT        = 3.0    # portfolio sizing stop-distance assumption
ENH_RISK_PER_TRADE = 0.01   # max capital risked per trade (scaled by conviction)
ENH_MAX_WEIGHT     = 0.20
ENH_LEVERAGE_CAP   = 2.50
ENH_TX_COST_BPS    = 2.0    # one-way transaction cost in bps

# ── Macro regime modulation ────────────────────────────────────────────────────
ENH_MACRO_ALPHA   = 0.5    # macro blend weight (0 = no effect, 1 = full blend)
ENH_MACRO_FAST    = 50     # SMA fast period for macro signals
ENH_MACRO_SLOW    = 200    # SMA slow period for macro signals
ENH_VOL_CAP_PCT   = 0.80   # FX vol percentile above which conviction is capped
ENH_VOL_DAMPEN    = 0.5    # conviction multiplier during high-vol regime


## 2. Strategy Run
Run the full Task 2 enhanced pipeline once with the control-panel parameters. Downstream diagnostics use these `pos_enh`, `wgt_enh`, and `port_r_enh` objects.


In [ ]:
from strategy_enhanced import build_signals as build_signals_enh, run_strategy as run_strategy_enh
from diagnostics import transaction_cost_sensitivity, sl_mult_grid_search, sar_grid_search

enhanced_kwargs = dict(
    # Layer 1
    filter_fast = ENH_FILTER_FAST,
    filter_slow = ENH_FILTER_SLOW,
    # Layer 2
    ema_window      = ENH_EMA_WINDOW,
    rsi_window      = ENH_RSI_WINDOW,
    rsi_long_level  = ENH_RSI_LONG_LEVEL,
    rsi_level       = ENH_RSI_LEVEL,
    rsi_flag_window = ENH_RSI_FLAG_WINDOW,
    rsi_short_cap   = ENH_RSI_SHORT_CAP,
    # Slot manager — SAR
    bb_window        = ENH_BB_WINDOW,
    bb_num_std       = ENH_BB_NUM_STD,
    atr_window       = ENH_ATR_WINDOW,
    sar_initial_mult = ENH_SAR_INITIAL_MULT,
    sar_af_start     = ENH_SAR_AF_START,
    sar_af_step      = ENH_SAR_AF_STEP,
    sar_af_max       = ENH_SAR_AF_MAX,
    sar_grace_period = ENH_SAR_GRACE_PERIOD,
    # Portfolio — conviction-weighted sizing
    sl_mult        = ENH_SL_MULT,
    risk_per_trade = ENH_RISK_PER_TRADE,
    max_weight     = ENH_MAX_WEIGHT,
    leverage_cap   = ENH_LEVERAGE_CAP,
    # Macro regime data
    external      = external,
    macro_alpha   = ENH_MACRO_ALPHA,
    macro_fast    = ENH_MACRO_FAST,
    macro_slow    = ENH_MACRO_SLOW,
    vol_cap_pct   = ENH_VOL_CAP_PCT,
    vol_dampen    = ENH_VOL_DAMPEN,
)

pos_enh, wgt_enh, port_r_enh = run_strategy_enh(
    prices,
    returns = returns,
    tx_cost_bps = ENH_TX_COST_BPS,
    **enhanced_kwargs,
)

pos_enh_gross, wgt_enh_gross, port_r_enh_gross = run_strategy_enh(
    prices,
    returns = returns,
    tx_cost_bps = 0.0,
    **enhanced_kwargs,
)

print(f'Enhanced net returns use tx_cost_bps = {ENH_TX_COST_BPS:g}')
print('Gross no-cost returns are available as port_r_enh_gross')


## 3. Signal Diagnostics
The next cells inspect the enhanced signal stack before evaluating performance.

### 3.1 Layer 1 Regime Filter
Measure the binary enhanced regime filter by commodity.


In [ ]:
from signals.signal_layer1_enhanced import layer1_signal as l1_enh_fn, regime_strength

l1_strength = regime_strength(prices, fast=ENH_FILTER_FAST, slow=ENH_FILTER_SLOW)
l1_enh      = l1_enh_fn(prices, filter_fast=ENH_FILTER_FAST, filter_slow=ENH_FILTER_SLOW)

print(f'prices      : {prices.shape}')
print(f'l1_strength : {l1_strength.shape}  — continuous [-1, +1]  (10 lookbacks: 20–110d)')
print(f'l1_enh      : {l1_enh.shape}   — binary {{-1, 0, +1, NaN}}')
print()

total = len(l1_enh)
counts_df = pd.DataFrame({
    'Long (+1)'       : (l1_enh ==  1).sum(),
    'Neutral (0)'     : (l1_enh ==  0).sum(),
    'Short (-1)'      : (l1_enh == -1).sum(),
    'NaN (warmup)'    : l1_enh.isna().sum(),
    '% Long'          : ((l1_enh ==  1).sum() / total * 100).round(1),
    '% Neutral'       : ((l1_enh ==  0).sum() / total * 100).round(1),
    '% Short'         : ((l1_enh == -1).sum() / total * 100).round(1),
    'Mean |strength|' : l1_strength.abs().mean().round(3),
})
counts_df.index.name = 'Commodity'
display(counts_df)


### 3.2 Layer 1 Regime Strength Chart
Visualise the continuous conviction grid used for position sizing.


In [ ]:
LEVELS = [(0.2, 0.06), (0.4, 0.09), (0.6, 0.13), (0.8, 0.17), (1.0, 0.22)]
REGIME_COMDTIES = ['BRENT CRUDE', "COFFEE 'C'", 'COPPER FUTURE']

fig, axes = plt.subplots(3, 1, figsize=(16, 13), sharex=False)
fig.suptitle(
    f'Layer 1 Regime Strength (Enhanced) — Multi-Lookback SMA Slope Aggregation\n'
    f'SMA{ENH_FILTER_FAST}/SMA{ENH_FILTER_SLOW}  |  Lookbacks: 20–110d (10 windows)\n'
    f'Gradient intensity = fraction of lookbacks agreeing with trend direction',
    fontweight='bold', fontsize=11
)

for ax, col in zip(axes, REGIME_COMDTIES):
    p     = prices[col]
    stren = l1_strength[col]
    sig   = l1_enh[col]

    sma_fast = p.rolling(ENH_FILTER_FAST).mean()
    sma_slow = p.rolling(ENH_FILTER_SLOW).mean()

    ax.plot(p.index, p.values,        color='black',      lw=1.0, zorder=3)
    ax.plot(p.index, sma_fast.values, color='steelblue',  lw=0.9, alpha=0.8, label=f'SMA{ENH_FILTER_FAST}')
    ax.plot(p.index, sma_slow.values, color='darkorange', lw=0.9, alpha=0.8, label=f'SMA{ENH_FILTER_SLOW}')

    for thresh, alpha in LEVELS:
        ax.fill_between(p.index, 0, 1,
                        where=(stren >= thresh),
                        transform=ax.get_xaxis_transform(),
                        color='#2ecc71', alpha=alpha, lw=0, zorder=1)
        ax.fill_between(p.index, 0, 1,
                        where=(stren <= -thresh),
                        transform=ax.get_xaxis_transform(),
                        color='#e74c3c', alpha=alpha, lw=0, zorder=1)

    n_total  = sig.notna().sum()
    pct_bull = (sig ==  1).sum() / n_total * 100
    pct_bear = (sig == -1).sum() / n_total * 100
    mean_abs = stren.abs().mean()

    ax.set_title(
        f'{col}   |   Bullish {pct_bull:.1f}%   Bearish {pct_bear:.1f}%   '
        f'Mean |strength| = {mean_abs:.2f}',
        fontsize=10, loc='left'
    )
    ax.legend(fontsize=8, loc='upper left')
    ax.set_ylabel('Price')
    ax.spines[['top', 'right']].set_visible(False)

fig.tight_layout()
plt.show()


### 3.3 Layer 2 Entry Signals
Count enhanced entry events, including the stricter short RSI trigger. The short RSI cap is checked later at execution time.


In [ ]:
from signals.signal_layer2_enhanced import layer2_signal as l2_enh_fn

l2_enh = l2_enh_fn(prices, layer1=l1_enh,
                   ema_window=ENH_EMA_WINDOW, rsi_window=ENH_RSI_WINDOW,
                   rsi_long_level=ENH_RSI_LONG_LEVEL, rsi_level=ENH_RSI_LEVEL,
                   rsi_flag_window=ENH_RSI_FLAG_WINDOW)

REGIME_COMDTIES = ['BRENT CRUDE', "COFFEE 'C'", 'COPPER FUTURE']

fig, axes = plt.subplots(3, 1, figsize=(16, 13), sharex=False)
fig.suptitle(
    f'Enhanced Layer 1 Regime + Layer 2 Entry Signals\n'
    f'SMA{ENH_FILTER_FAST}/SMA{ENH_FILTER_SLOW}  |  '
    f'EMA{ENH_EMA_WINDOW}  RSI_long>{ENH_RSI_LONG_LEVEL}  RSI_short<{ENH_RSI_LEVEL}  exec_cap={ENH_RSI_SHORT_CAP}\n'
    f'Green=Bullish  Red=Bearish  ▲=Long entry  ▽=Short entry',
    fontweight='bold', fontsize=11
)

for ax, col in zip(axes, REGIME_COMDTIES):
    p    = prices[col]
    sig1 = l1_enh[col]
    sig2 = l2_enh[col]

    ax.plot(p.index, p.values, color='black', lw=1.0, zorder=3)
    ax.fill_between(p.index, 0, 1, where=(sig1 == 1),
                    transform=ax.get_xaxis_transform(),
                    color='#2ecc71', alpha=0.20, lw=0, zorder=1)
    ax.fill_between(p.index, 0, 1, where=(sig1 == -1),
                    transform=ax.get_xaxis_transform(),
                    color='#e74c3c', alpha=0.20, lw=0, zorder=1)

    long_days  = sig2[sig2 ==  1].index
    short_days = sig2[sig2 == -1].index
    ax.scatter(long_days,  p.loc[long_days],  marker='^', color='#27ae60', s=60, zorder=5, label='Long entry')
    ax.scatter(short_days, p.loc[short_days], marker='v', color='#c0392b', s=60, zorder=5, label='Short entry')

    n_total  = sig1.notna().sum()
    pct_bull = (sig1 ==  1).sum() / n_total * 100
    pct_bear = (sig1 == -1).sum() / n_total * 100
    ax.set_title(
        f'{col}   |   Bullish {pct_bull:.1f}%  Bearish {pct_bear:.1f}%   |   '
        f'Long: {len(long_days)}   Short: {len(short_days)}',
        fontsize=10, loc='left'
    )
    ax.set_ylabel('Price')
    ax.legend(fontsize=8, loc='upper left', ncol=2)
    ax.spines[['top', 'right']].set_visible(False)

fig.tight_layout()
plt.show()


## 4. Position And Risk Diagnostics
The next cells inspect realised positions, SAR-managed exits, and macro effects after the enhanced pipeline runs.

### 4.1 Position Grid Analysis
Summarise long, short, and flat exposure after applying execution-time RSI cap, SAR exits, and no-clustering.


In [ ]:
import matplotlib.colors as mcolors

cmap = mcolors.ListedColormap(['firebrick', 'whitesmoke', 'steelblue'])
norm = mcolors.BoundaryNorm([-1.5, -0.5, 0.5, 1.5], cmap.N)
step = max(1, len(pos_enh) // 10)

fig, ax = plt.subplots(figsize=(16, 7))
ax.imshow(pos_enh.T.values, aspect='auto', cmap=cmap, norm=norm, interpolation='none')
ax.set_yticks(range(len(pos_enh.columns)))
ax.set_yticklabels(pos_enh.columns, fontsize=8)
ax.set_xticks(range(0, len(pos_enh), step))
ax.set_xticklabels([str(d.date()) for d in pos_enh.index[::step]], rotation=45, fontsize=7)
ax.set_title('Position Grid (Enhanced) — Blue=Long | Red=Short | White=Flat', fontweight='bold')
fig.tight_layout()
plt.show()

print(position_summary(pos_enh))


### 4.2 Macro Impact Analysis
Compare raw and macro-adjusted conviction to confirm macro is acting as a sizing overlay rather than a direction signal.


In [ ]:
from signals.macro_signal import blend_regime_strength, apply_vol_cap
from signals.signal_layer1_enhanced import regime_strength as _raw_strength

# Raw commodity-only regime strength (no macro)
raw_strength = _raw_strength(prices,
                              fast=ENH_FILTER_FAST, slow=ENH_FILTER_SLOW)

# Macro-adjusted strength (same as used inside run_strategy_enh)
adj_strength = blend_regime_strength(raw_strength, m_score, alpha=ENH_MACRO_ALPHA)
adj_strength = apply_vol_cap(adj_strength, v_pct,
                             vol_cap_pct=ENH_VOL_CAP_PCT, vol_dampen=ENH_VOL_DAMPEN)

raw_flat = raw_strength.values.flatten()
adj_flat = adj_strength.values.flatten()
mask     = ~(np.isnan(raw_flat) | np.isnan(adj_flat) | (raw_flat == 0))

diff = adj_flat[mask] - raw_flat[mask]
pct_boosted = (diff > 0.01).mean() * 100
pct_damped  = (diff < -0.01).mean() * 100
pct_capped  = ((v_pct.reindex(prices.index) > ENH_VOL_CAP_PCT).fillna(False)).mean() * 100

print(f'Regime-active bars:  {mask.sum():,}')
print(f'  Boosted by macro : {pct_boosted:5.1f}%  (macro agreement → larger size)')
print(f'  Damped by macro  : {pct_damped:5.1f}%  (macro opposition → smaller size)')
print(f'  High-vol cap days: {pct_capped:5.1f}%  (EURUSD 3M ATM vol > {ENH_VOL_CAP_PCT:.0%} pct)')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.hist(raw_flat[mask], bins=50, alpha=0.55, color='steelblue',  label='Raw (commodity only)')
ax.hist(adj_flat[mask], bins=50, alpha=0.55, color='darkorange', label='Macro-adjusted')
ax.set_xlabel('Regime strength')
ax.set_ylabel('Frequency')
ax.set_title('Distribution of Regime Conviction\n(active positions only)', fontweight='bold')
ax.legend(fontsize=9)
ax.spines[['top', 'right']].set_visible(False)

ax = axes[1]
ax.scatter(raw_flat[mask], adj_flat[mask], alpha=0.05, s=2, color='steelblue')
lims = [min(raw_flat[mask].min(), adj_flat[mask].min()),
        max(raw_flat[mask].max(), adj_flat[mask].max())]
ax.plot(lims, lims, 'k--', lw=0.8, label='No change')
ax.set_xlabel('Raw conviction')
ax.set_ylabel('Macro-adjusted conviction')
ax.set_title('Raw vs Macro-Adjusted Conviction\n(above diagonal = boosted, below = damped)', fontweight='bold')
ax.legend(fontsize=9)
ax.spines[['top', 'right']].set_visible(False)

fig.tight_layout()
plt.show()


## 5. Performance And Robustness
Compare Task 2 against the Task 1 baseline and the equal-weight factor using net returns after the configured transaction cost.

### 5.1 Summary Statistics
Report annualised return, volatility, Sharpe, drawdown, turnover, and trade counts.


In [ ]:
stats_enh = pd.concat([
    performance_stats(port_r_enh, label=f'Enhanced Strategy Net ({ENH_TX_COST_BPS:g} bps)',
                      factor=factor, weights=wgt_enh, positions=pos_enh),
    performance_stats(port_r_enh_gross, label='Enhanced Strategy Gross (0 bps)',
                      factor=factor, weights=wgt_enh_gross, positions=pos_enh_gross),
    performance_stats(port_r, label='Baseline Strategy',
                      factor=factor, weights=weights, positions=positions),
    performance_stats(factor.reindex(port_r_enh.index).dropna(), label='EW Factor'),
], axis=1).T

stats_enh


### 5.2 Transaction Cost Sensitivity
Re-run the enhanced strategy on training data under different one-way transaction-cost assumptions.


In [ ]:
tc_sensitivity = transaction_cost_sensitivity(
    prices,
    returns,
    cost_bps=(0.0, 2.0, 5.0, 10.0),
    strategy_kwargs=enhanced_kwargs,
)

tc_display = tc_sensitivity.copy()
for col in ['ann_return_net', 'ann_vol_net', 'max_drawdown', 'ann_turnover',
            'ann_return_gross', 'ann_cost', 'long_contribution_ann', 'short_contribution_ann']:
    if col in tc_display:
        tc_display[col] = (tc_display[col] * 100).round(2)
for col in ['sharpe_net', 'sharpe_gross']:
    if col in tc_display:
        tc_display[col] = tc_display[col].round(3)

display(tc_display)


### 5.3 SL Multiplier Training Grid
Run the predefined training-only `sl_mult` sizing grid. This diagnostic changes only the portfolio sizing stop-distance assumption; SAR exits, RSI filters, macro settings, and transaction costs stay fixed.


In [ ]:
sl_mult_grid_results = sl_mult_grid_search(
    prices,
    returns,
    strategy_kwargs=enhanced_kwargs,
    tx_cost_bps=ENH_TX_COST_BPS,
)

sl_mult_grid_path = Path('sl_mult_grid_results.csv')
sl_mult_grid_results.to_csv(sl_mult_grid_path, index=False)

sl_cols = [
    'passes_basic_filter', 'sharpe_net', 'sharpe_gross', 'max_drawdown',
    'entries', 'short_entries', 'ann_turnover', 'ann_cost',
    'long_contribution_ann', 'short_contribution_ann', 'sl_mult',
    'baseline_sharpe_net', 'baseline_max_drawdown', 'baseline_entries',
    'baseline_ann_turnover', 'baseline_ann_cost',
]

display(sl_mult_grid_results[sl_cols])
print(f'Saved full SL multiplier grid to {sl_mult_grid_path}')


### 5.4 SAR Training Grid
Run the predefined training-only SAR grid. This diagnostic ranks candidates by net Sharpe and robustness filters, but the top row is not automatically the final choice.


In [ ]:
sar_grid_results = sar_grid_search(
    prices,
    returns,
    strategy_kwargs=enhanced_kwargs,
    tx_cost_bps=ENH_TX_COST_BPS,
)

sar_grid_path = Path('sar_grid_results.csv')
sar_grid_results.to_csv(sar_grid_path, index=False)

sar_cols = [
    'passes_basic_filter', 'sharpe_net', 'sharpe_gross', 'max_drawdown',
    'entries', 'short_entries', 'ann_turnover', 'ann_cost',
    'sar_initial_mult', 'sar_af_start', 'sar_af_step', 'sar_af_max', 'sar_grace_period',
    'baseline_sharpe_net', 'baseline_max_drawdown', 'baseline_entries',
]

display(sar_grid_results[sar_cols].head(20))
print(f'Saved full SAR grid to {sar_grid_path}')


### 5.5 Rolling Sharpe
Print full-period Sharpe sanity checks, then plot one-year rolling Sharpe on a common date index.


In [ ]:
def _full_period_sharpe(r):
    r = r.dropna()
    return (r.mean() * 252) / (r.std() * np.sqrt(252))

enh_short_entries = int(((pos_enh == -1) & (pos_enh.shift(1).fillna(0) == 0)).sum().sum())

print(f"Enhanced net full-period Sharpe   : {_full_period_sharpe(port_r_enh):.3f}  ({ENH_TX_COST_BPS:g} bps)")
print(f"Enhanced gross full-period Sharpe : {_full_period_sharpe(port_r_enh_gross):.3f}  (0 bps)")
print(f"Baseline full-period Sharpe       : {_full_period_sharpe(port_r.reindex(port_r_enh.index)):.3f}")
print(f"EW Factor full-period Sharpe      : {_full_period_sharpe(factor.reindex(port_r_enh.index)):.3f}")
print(f"Enhanced short entries            : {enh_short_entries}")

rolling_sharpe_df = plot_rolling_sharpe(
    {'Enhanced Strategy Net': port_r_enh,
     'Enhanced Strategy Gross': port_r_enh_gross.reindex(port_r_enh.index),
     'Baseline Strategy': port_r.reindex(port_r_enh.index),
     'EW Factor': factor.reindex(port_r_enh.index)},
    window=252,
)


### 5.6 Cumulative Returns
Plot log cumulative returns for enhanced Task 2, Task 1 baseline, and the equal-weight commodity factor.


In [ ]:
cum_enh    = (1 + port_r_enh.dropna()).cumprod()
cum_base   = (1 + port_r.reindex(port_r_enh.index).dropna()).cumprod()
cum_factor = (1 + factor.reindex(port_r_enh.index).dropna()).cumprod()

fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(cum_enh.index,    cum_enh.values,    color='steelblue',  lw=2.0, label='Enhanced Strategy')
ax.plot(cum_base.index,   cum_base.values,   color='darkorange', lw=1.5, linestyle='--', label='Baseline Strategy')
ax.plot(cum_factor.index, cum_factor.values, color='black',      lw=1.2, linestyle=':',  label='EW Factor')
ax.axhline(1, color='grey', lw=0.5)
ax.set_yscale('log')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.1f}x'))
ax.set_title('Cumulative Return — Enhanced vs Baseline vs EW Factor', fontsize=13, fontweight='bold')
ax.set_ylabel('Cumulative Return (log scale)')
ax.legend(fontsize=9)
ax.spines[['top', 'right']].set_visible(False)
fig.tight_layout()
plt.show()


### 5.7 Sector Contribution
Break enhanced returns into sector-level contribution and show final position exposure.


In [ ]:
sc_enh = sector_contribution(wgt_enh, returns, assets)

sc_enh_summary = pd.DataFrame({
    'Ann. Contribution %' : (sc_enh.mean() * 252 * 100).round(2),
    'Ann. Vol %'          : (sc_enh.std()  * np.sqrt(252) * 100).round(2),
    'Sharpe'              : ((sc_enh.mean() * 252) / (sc_enh.std() * np.sqrt(252))).round(3),
})
display(sc_enh_summary)

sector_colours = {'Agri & livestock': '#2e8b57', 'Energy': '#cc4400', 'Metals': '#4169e1'}
fig, ax = plt.subplots(figsize=(13, 4))
for col in sc_enh.cumsum().columns:
    ax.plot(sc_enh.cumsum().index, sc_enh.cumsum()[col],
            color=sector_colours.get(col, 'grey'), lw=1.8, label=col)
ax.axhline(0, color='grey', lw=0.5)
ax.set_title('Cumulative Return Contribution by Sector (Enhanced)', fontweight='bold')
ax.legend(fontsize=9)
ax.spines[['top', 'right']].set_visible(False)
fig.tight_layout()
plt.show()


## 6. Final Summary Notes
Task 2 keeps the Task 1 trend-following identity while adding enhanced conviction, macro sizing, execution-time short RSI filtering, Parabolic SAR exits, and transaction costs. All tuning diagnostics in this notebook are training-only. Freeze selected parameters before running any final holdout evaluation.
